In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install human-eval datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 7.3 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login("hf_token")

In [4]:
!pip install -U bitsandbytes>=0.46.1

In [5]:
import torch
import random
import numpy as np
from math import comb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
from human_eval.execution import check_correctness

# ---------------- CONFIG ----------------
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"
ADAPTER_MODEL = "pradip777/llama-2-PCG"

MAX_NEW_TOKENS = 256
NUM_SAMPLES_PER_PROBLEM = 5   # required for pass@5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- LOAD MODEL ----------------
print("🚀 Loading model...")

if DEVICE == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    device_map = "auto"
else:
    bnb_config = None
    device_map = {"": "cpu"}

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map=device_map
)

model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL)
model.eval()

print("✅ Model loaded")

# ---------------- LOAD DATASET ----------------
dataset = load_dataset("openai_humaneval")["test"]
total_problems = len(dataset)
print(f"Total Problems: {total_problems}")

# ---------------- PASS@K STORAGE ----------------
results_per_problem = []

# ---------------- GENERATE + EVALUATE ----------------
for idx, problem in enumerate(dataset):

    prompt = problem["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    correct_count = 0

    for sample_id in range(NUM_SAMPLES_PER_PROBLEM):

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.8,        # important for diversity
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        completion = generated[len(prompt):]
        completion = completion.split("\n\n")[0]

        try:
            result = check_correctness(problem, completion, timeout=5.0)
            if result["passed"]:
                correct_count += 1
        except:
            pass

    results_per_problem.append(correct_count)

    print(f"Problem {idx+1}/{total_problems} | Correct: {correct_count}/5")

# ---------------- PASS@K COMPUTATION ----------------

def compute_pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1 - (comb(n - c, k) / comb(n, k))

pass_scores = {}

for k in range(1, 6):
    score = 0
    for c in results_per_problem:
        score += compute_pass_at_k(NUM_SAMPLES_PER_PROBLEM, c, k)
    pass_scores[f"pass@{k}"] = score / total_problems

# ---------------- PRINT RESULTS ----------------
print("\n" + "="*60)
for k, v in pass_scores.items():
    print(f"{k}: {v:.4f}")
print("="*60)

🚀 Loading model...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

✅ Model loaded


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Total Problems: 164
Problem 1/164 | Correct: 0/5
Problem 2/164 | Correct: 0/5
Problem 3/164 | Correct: 4/5
Problem 4/164 | Correct: 0/5
Problem 5/164 | Correct: 4/5
Problem 6/164 | Correct: 0/5
Problem 7/164 | Correct: 0/5
Problem 8/164 | Correct: 5/5
Problem 9/164 | Correct: 2/5
Problem 10/164 | Correct: 0/5
Problem 11/164 | Correct: 0/5
Problem 12/164 | Correct: 0/5
Problem 13/164 | Correct: 1/5
Problem 14/164 | Correct: 1/5
Problem 15/164 | Correct: 1/5
Problem 16/164 | Correct: 3/5
Problem 17/164 | Correct: 0/5
Problem 18/164 | Correct: 1/5
Problem 19/164 | Correct: 1/5
Problem 20/164 | Correct: 0/5
Problem 21/164 | Correct: 0/5
Problem 22/164 | Correct: 0/5
Problem 23/164 | Correct: 4/5
Problem 24/164 | Correct: 5/5
Problem 25/164 | Correct: 0/5
Problem 26/164 | Correct: 0/5
Problem 27/164 | Correct: 0/5
Problem 28/164 | Correct: 1/5
Problem 29/164 | Correct: 4/5
Problem 30/164 | Correct: 5/5
Problem 31/164 | Correct: 5/5
Problem 32/164 | Correct: 1/5
Problem 33/164 | Correct: 0/5